# 🦷 Periodontal Bone Loss AI — Google Colab Training

**Complete end-to-end training pipeline for all 3 model architectures on FREE GPU.**

### ✅ What this notebook does:
1. Checks GPU availability
2. Clones your GitHub repository
3. Installs all dependencies
4. Mounts Google Drive to load your dataset
5. Runs data split (70/15/15 patient-level)
6. Trains EfficientNet-B0, ResNet-50, and CNN
7. Evaluates all models and runs ablation study
8. Downloads trained models back to your PC

---
**⚠️ IMPORTANT: Before running, go to `Runtime → Change runtime type → T4 GPU`**

## 🔷 STEP 1 — Check GPU

In [ ]:
# Check what GPU Colab gave us
import torch

print('='*50)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU Available: {gpu_name}')
    print(f'   VRAM: {gpu_mem:.1f} GB')
else:
    print('❌ No GPU found!')
    print('   Go to: Runtime → Change runtime type → T4 GPU')
    raise RuntimeError('GPU required. Please enable GPU runtime.')
print('='*50)

!nvidia-smi

## 🔷 STEP 2 — Clone Your GitHub Repository

In [ ]:
import os

# Clone the repo (or update if already cloned)
REPO_URL  = 'https://github.com/ramrajasekaran/periodontal_bone_loss_ai.git'
REPO_NAME = 'periodontal_bone_loss_ai'

if os.path.exists(REPO_NAME):
    print('📁 Repo already exists — pulling latest changes...')
    !cd {REPO_NAME} && git pull
else:
    print('📥 Cloning repository...')
    !git clone {REPO_URL}

# Set working directory to backend
os.chdir(f'/content/{REPO_NAME}')
print(f'\n✅ Working directory: {os.getcwd()}')
!ls -la

## 🔷 STEP 3 — Install Dependencies

In [ ]:
# Install all required packages
# (torch & torchvision are pre-installed on Colab with CUDA support)
print('📦 Installing dependencies...')
!pip install -q opencv-python-headless tqdm pandas matplotlib seaborn Pillow flask flask-cors streamlit

# Verify key installs
import torch, torchvision, cv2, pandas, numpy
print(f'\n✅ PyTorch     : {torch.__version__}  (CUDA: {torch.version.cuda})')
print(f'✅ TorchVision : {torchvision.__version__}')
print(f'✅ OpenCV      : {cv2.__version__}')
print(f'✅ Pandas      : {pandas.__version__}')
print(f'✅ NumPy       : {numpy.__version__}')
print('\n🎉 All dependencies installed!')

## 🔷 STEP 4 — Mount Google Drive (to access your dataset)

**Before running this cell:**
1. Upload your dataset folder to Google Drive
2. The folder should contain:
   - `meta_data.csv` — your labels file
   - `images/` folder — all your X-ray images

**Expected Drive folder structure:**
```
My Drive/
└── periodontal_dataset/
    ├── meta_data.csv
    └── images/
        ├── patient_image_000001_xxxx.jpg
        ├── patient_image_000002_xxxx.jpg
        └── ...
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 👇 CHANGE THIS to match your Google Drive folder path
DRIVE_DATASET_PATH = '/content/drive/MyDrive/periodontal_dataset'

import os
if os.path.exists(DRIVE_DATASET_PATH):
    print(f'✅ Dataset folder found: {DRIVE_DATASET_PATH}')
    !ls {DRIVE_DATASET_PATH}
else:
    print(f'❌ Dataset folder NOT found at: {DRIVE_DATASET_PATH}')
    print('   Please update DRIVE_DATASET_PATH to match your Drive folder name.')

## 🔷 STEP 5 — Copy Dataset into the Project

In [ ]:
import os, shutil

# Paths inside the cloned repo
BACKEND_DATA_DIR = '/content/periodontal_bone_loss_ai/backend/data'
os.makedirs(BACKEND_DATA_DIR, exist_ok=True)

# Copy meta_data.csv
src_meta = os.path.join(DRIVE_DATASET_PATH, 'meta_data.csv')
dst_meta = os.path.join(BACKEND_DATA_DIR, 'meta_data.csv')
shutil.copy2(src_meta, dst_meta)
print(f'✅ Copied meta_data.csv ({os.path.getsize(dst_meta):,} bytes)')

# Symlink or copy images folder (symlink is faster — avoids duplicating GBs of data)
src_images = os.path.join(DRIVE_DATASET_PATH, 'images')
dst_images = os.path.join(BACKEND_DATA_DIR, 'images')

if os.path.exists(dst_images):
    print('ℹ️  images/ link already exists')
elif os.path.exists(src_images):
    os.symlink(src_images, dst_images)
    print(f'✅ Linked images/ folder from Drive')
else:
    print(f'❌ images/ folder not found at: {src_images}')

# Quick check
n_images = len(os.listdir(dst_images)) if os.path.exists(dst_images) else 0
print(f'\n📊 Dataset summary:')
print(f'   meta_data.csv : {dst_meta}')
print(f'   images/       : {n_images} files')

## 🔷 STEP 6 — Generate 70/15/15 Data Split

In [ ]:
# Run data_split.py — creates train.csv, val.csv, test.csv, class_mapping.json
os.chdir('/content/periodontal_bone_loss_ai')

print('🔀 Running patient-level 70/15/15 stratified split...')
!python backend/src/data_split.py

# Verify the output files
import pandas as pd
data_dir = 'backend/data'
for split in ['train', 'val', 'test']:
    csv_path = f'{data_dir}/{split}.csv'
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        print(f'✅ {split}.csv — {len(df)} rows')
    else:
        print(f'❌ {split}.csv not found!')

## 🔷 STEP 7 — Train EfficientNet-B0 (Best Model)

This is the strongest model. Train it first.

In [ ]:
print('🚀 Training EfficientNet-B0...')
print('   Expected time: ~15-25 min on T4 GPU')
print('='*60)

!python backend/src/train.py \
    --model efficientnet \
    --epochs 50 \
    --batch_size 16 \
    --experiment 1

print('\n✅ EfficientNet-B0 training complete!')

## 🔷 STEP 8 — Train ResNet-50

In [ ]:
print('🚀 Training ResNet-50...')
print('   Expected time: ~15-25 min on T4 GPU')
print('='*60)

!python backend/src/train.py \
    --model resnet50 \
    --epochs 50 \
    --batch_size 16 \
    --experiment 1

print('\n✅ ResNet-50 training complete!')

## 🔷 STEP 9 — Train Custom CNN

In [ ]:
print('🚀 Training Custom CNN...')
print('   Expected time: ~10-15 min on T4 GPU')
print('='*60)

!python backend/src/train.py \
    --model cnn \
    --epochs 50 \
    --batch_size 32 \
    --experiment 1

print('\n✅ CNN training complete!')

## 🔷 STEP 10 — Evaluate All Models

In [ ]:
print('📊 Evaluating all models on the held-out test set...')
print('='*60)

!python backend/src/evaluate.py --model all

print('\n✅ Evaluation complete! Check backend/results/tables/')

## 🔷 STEP 11 — Run Ablation Study

In [ ]:
print('🔬 Running Ablation Study (Image-only vs Tabular-only vs Multimodal)...')
print('='*60)

!python backend/src/ablation.py --mode full

print('\n✅ Ablation study complete!')

## 🔷 STEP 12 — View Results Summary

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import json, os

# --- Model Comparison Table ---
comparison_csv = 'backend/results/tables/model_comparison.csv'
if os.path.exists(comparison_csv):
    df = pd.read_csv(comparison_csv)
    print('📊 MODEL COMPARISON:')
    print(df.to_string(index=False))
else:
    print('⚠️  model_comparison.csv not found — run Step 10 first.')

print('\n' + '='*60)

# --- Training Curves ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = {'efficientnet': '#4CAF50', 'resnet50': '#2196F3', 'cnn': '#FF9800'}

for i, model_name in enumerate(['efficientnet', 'resnet50', 'cnn']):
    hist_path = f'backend/results/history_{model_name}_exp1.json'
    if os.path.exists(hist_path):
        with open(hist_path) as f:
            history = json.load(f)
        ax = axes[i]
        ax.plot(history['train_loss'], label='Train Loss', color=colors[model_name], linewidth=2)
        ax.plot(history['val_loss'],   label='Val Loss',   color=colors[model_name], linewidth=2, linestyle='--')
        ax2 = ax.twinx()
        ax2.plot(history['val_f1'], label='Val F1', color='red', linewidth=2, linestyle=':')
        ax.set_title(f'{model_name.upper()} Training Curves', fontsize=13, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax2.set_ylabel('Macro F1', color='red')
        ax.legend(loc='upper left')
        ax2.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
    else:
        axes[i].text(0.5, 0.5, f'{model_name}\nnot trained yet',
                     ha='center', va='center', transform=axes[i].transAxes)

plt.suptitle('Periodontal Bone Loss AI — Training Results', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('backend/results/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Training curves saved to backend/results/training_curves.png')

## 🔷 STEP 13 — Save Everything to Google Drive & Download

In [ ]:
import shutil, os

# Save trained models and results to Google Drive
DRIVE_OUTPUT = '/content/drive/MyDrive/periodontal_results'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Copy models
models_src = 'backend/models'
models_dst = os.path.join(DRIVE_OUTPUT, 'models')
if os.path.exists(models_src):
    shutil.copytree(models_src, models_dst, dirs_exist_ok=True)
    print(f'✅ Models saved to Google Drive: {models_dst}')

# Copy results
results_src = 'backend/results'
results_dst = os.path.join(DRIVE_OUTPUT, 'results')
if os.path.exists(results_src):
    shutil.copytree(results_src, results_dst, dirs_exist_ok=True)
    print(f'✅ Results saved to Google Drive: {results_dst}')

# Copy configs
configs_src = 'backend/configs'
configs_dst = os.path.join(DRIVE_OUTPUT, 'configs')
if os.path.exists(configs_src):
    shutil.copytree(configs_src, configs_dst, dirs_exist_ok=True)
    print(f'✅ Configs saved to Google Drive: {configs_dst}')

print(f'\n🎉 All outputs saved to: {DRIVE_OUTPUT}')
!ls {DRIVE_OUTPUT}

In [ ]:
# OPTIONAL: Download models directly to your browser
from google.colab import files
import os

models_dir = 'backend/models'
if os.path.exists(models_dir):
    for f in os.listdir(models_dir):
        if f.endswith('.pth'):
            filepath = os.path.join(models_dir, f)
            size_mb = os.path.getsize(filepath) / 1e6
            print(f'📥 Downloading {f} ({size_mb:.1f} MB)...')
            files.download(filepath)
else:
    print('No models found — run training steps first.')

## 🔷 STEP 14 — BONUS: Experiment 2 (WeightedRandomSampler)

Run this AFTER Experiment 1 to compare performance with balanced sampling.

In [ ]:
# Experiment 2: WeightedRandomSampler for balanced batches
print('🧪 Running Experiment 2 (WeightedRandomSampler)...')

!python backend/src/train.py --model efficientnet --epochs 50 --batch_size 16 --experiment 2
!python backend/src/train.py --model resnet50     --epochs 50 --batch_size 16 --experiment 2
!python backend/src/train.py --model cnn          --epochs 50 --batch_size 32 --experiment 2

print('\n✅ Experiment 2 complete!')